In [ ]:
-- 실행 컨텍스트 설정
USE ROLE ACCOUNTADMIN;
USE DATABASE DEMO;
USE SCHEMA ML_DEMO;


# Module 6: ML Jobs & Pipeline Orchestration

ML 학습을 Compute Pool에서 실행하고, Task Graph(DAG)로 자동화합니다.

### 학습 목표
- `@remote`로 Compute Pool에서 학습 코드 실행
- Task Graph(DAG)로 ML 파이프라인 스케줄링


### 핵심 개념

| 개념 | 설명 |
|------|------|
| **ML Jobs (`@remote`)** | Python 함수를 Compute Pool(CPU/GPU 컨테이너)에서 원격 실행. 비동기 |
| **Compute Pool** | Snowflake 관리형 컨테이너 클러스터. GPU 사용 가능, 자동 스케일 |
| **Task Graph (DAG)** | Snowflake 네이티브 파이프라인 오케스트레이션. 스케줄 + 의존성 관리 |
| **Stored Procedure** | Task는 SQL만 실행하므로 Python 학습 로직을 SP로 래핑하여 호출 |


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F
from snowflake.ml.jobs import remote
from snowflake.ml.registry import Registry

session = get_active_session()
session.use_database('DEMO')
session.use_schema('ML_DEMO')
print('Session ready')


## 1. ML Jobs (`@remote`)

로컬/Notebook 함수를 Compute Pool에서 실행합니다.


In [ ]:
@remote(session=session, compute_pool='SYSTEM_COMPUTE_POOL_CPU',
        stage_name='@DEMO.ML_DEMO.ML_STAGE')
def train_model(config: dict) -> dict:
    """Compute Pool에서 실행되는 학습 함수"""
    from snowflake.snowpark.context import get_active_session
    from snowflake.ml.modeling.xgboost import XGBRegressor
    from snowflake.ml.modeling.pipeline import Pipeline
    from snowflake.ml.modeling.preprocessing import StandardScaler, OrdinalEncoder
    from snowflake.ml import dataset
    import numpy as np
    from sklearn.metrics import mean_squared_error

    sess = get_active_session()
    ds = dataset.load_dataset(sess, 'customer_ltv_training', 'v1')
    df = ds.read.to_snowpark_dataframe()
    train_df, test_df = df.random_split([0.8, 0.2], seed=42)

    pipeline = Pipeline(steps=[
        ('encoder', OrdinalEncoder(
            input_cols=['C_MKTSEGMENT'], output_cols=['C_MKTSEGMENT'],
            handle_unknown='use_encoded_value', unknown_value=-1)),
        ('scaler', StandardScaler(
            input_cols=['C_ACCTBAL','TOTAL_ORDERS','AVG_ORDER_VALUE','TOTAL_REVENUE',
                       'AVG_DISCOUNT','AVG_QUANTITY','DAYS_SINCE_LAST_ORDER','C_NATIONKEY'],
            output_cols=['C_ACCTBAL','TOTAL_ORDERS','AVG_ORDER_VALUE','TOTAL_REVENUE',
                       'AVG_DISCOUNT','AVG_QUANTITY','DAYS_SINCE_LAST_ORDER','C_NATIONKEY'])),
        ('regressor', XGBRegressor(
            input_cols=['C_MKTSEGMENT','C_ACCTBAL','TOTAL_ORDERS','AVG_ORDER_VALUE',
                       'TOTAL_REVENUE','AVG_DISCOUNT','AVG_QUANTITY','DAYS_SINCE_LAST_ORDER','C_NATIONKEY'],
            label_cols=['FUTURE_12M_REVENUE'], output_cols=['PREDICTED_LABEL'],
            n_estimators=config['n_estimators'], max_depth=config['max_depth'],
            learning_rate=config['learning_rate'], random_state=42))
    ])
    pipeline.fit(train_df)

    preds = pipeline.predict(test_df).select('FUTURE_12M_REVENUE','PREDICTED_LABEL').to_pandas()
    rmse = float(np.sqrt(mean_squared_error(preds['FUTURE_12M_REVENUE'], preds['PREDICTED_LABEL'])))
    return {'rmse': rmse, 'config': config}

print('train_model function defined (will run on Compute Pool)')


In [ ]:
# ML Job 실행
job = train_model({'n_estimators': 150, 'max_depth': 5, 'learning_rate': 0.08})
print(f'Job submitted: {job.id}')
print(f'Status: {job.status}')

# 결과 대기
result = job.result()
print(f'Result: RMSE={result["rmse"]:,.0f}')


## 2. Task Graph (DAG)

ML 파이프라인을 자동화합니다. Task는 SQL만 실행하므로 학습 로직은 Stored Procedure로 래핑합니다.


In [ ]:
# Stored Procedure 생성 (DAG에서 호출할 SP 3개)

# SP 1: 피처 상태 확인
session.sql("""
CREATE OR REPLACE PROCEDURE DEMO.ML_DEMO.PREPARE_FEATURES()
RETURNS STRING
LANGUAGE SQL
AS
BEGIN
    LET row_count INTEGER := (SELECT COUNT(*) FROM DEMO.ML_DEMO.CUSTOMER_LTV_FEATURES$1);
    RETURN 'Features ready: ' || row_count || ' rows in Managed FV';
END;
""").collect()

# SP 2: 학습 모델 존재 확인
session.sql("""
CREATE OR REPLACE PROCEDURE DEMO.ML_DEMO.TRAIN_MODEL_PROC()
RETURNS STRING
LANGUAGE SQL
AS
BEGIN
    LET model_exists BOOLEAN := (SELECT COUNT(*) > 0 FROM INFORMATION_SCHEMA.MODELS WHERE MODEL_NAME = 'CUSTOMER_LTV_PREDICTOR');
    IF (model_exists) THEN
        RETURN 'Model exists: CUSTOMER_LTV_PREDICTOR';
    ELSE
        RETURN 'ERROR: Model not found';
    END IF;
END;
""").collect()

# SP 3: 배치 스코어링 (실제 추론 실행)
session.sql("""
CREATE OR REPLACE PROCEDURE DEMO.ML_DEMO.SCORE_CUSTOMERS()
RETURNS STRING
LANGUAGE SQL
AS
BEGIN
    CREATE OR REPLACE TABLE DEMO.ML_DEMO.CUSTOMER_LTV_SCORES AS
    WITH latest AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY C_CUSTKEY ORDER BY FEATURE_DATE DESC) AS rn
        FROM DEMO.ML_DEMO.CUSTOMER_LTV_FEATURES$1
    )
    SELECT
        C_CUSTKEY,
        C_MKTSEGMENT,
        TOTAL_ORDERS,
        AVG_ORDER_VALUE,
        TOTAL_REVENUE,
        MODEL(DEMO.ML_DEMO.CUSTOMER_LTV_PREDICTOR)!PREDICT(
            C_MKTSEGMENT, C_ACCTBAL, TOTAL_ORDERS, AVG_ORDER_VALUE,
            TOTAL_REVENUE, AVG_DISCOUNT, AVG_QUANTITY,
            DAYS_SINCE_LAST_ORDER, C_NATIONKEY
        ):PREDICTED_LABEL::FLOAT AS PREDICTED_LTV,
        CURRENT_TIMESTAMP() AS SCORED_AT
    FROM latest WHERE rn = 1;
    RETURN 'Scoring complete: ' || (SELECT COUNT(*) FROM DEMO.ML_DEMO.CUSTOMER_LTV_SCORES) || ' rows';
END;
""").collect()

print('3 Stored Procedures created:')
print('  - PREPARE_FEATURES (피처 상태 확인)')
print('  - TRAIN_MODEL_PROC (학습 모델 존재 확인)')
print('  - SCORE_CUSTOMERS (배치 스코어링 실행)')


In [ ]:
from snowflake.core.task.dagv1 import DAG, DAGTask, DAGOperation
from snowflake.core._common import CreateMode
from snowflake.core import Root
from datetime import timedelta

root = Root(session)

# 기존 DAG suspend 후 삭제
session.sql('ALTER TASK IF EXISTS DEMO.ML_DEMO.ML_TRAINING_DAG SUSPEND').collect()
session.sql('DROP TASK IF EXISTS DEMO.ML_DEMO.ML_TRAINING_DAG$SCORE_CUSTOMERS').collect()
session.sql('DROP TASK IF EXISTS DEMO.ML_DEMO.ML_TRAINING_DAG$TRAIN_MODEL').collect()
session.sql('DROP TASK IF EXISTS DEMO.ML_DEMO.ML_TRAINING_DAG$PREPARE_FEATURES').collect()
session.sql('DROP TASK IF EXISTS DEMO.ML_DEMO.ML_TRAINING_DAG$MODEL_TRAIN').collect()
session.sql('DROP TASK IF EXISTS DEMO.ML_DEMO.ML_TRAINING_DAG$MONITOR_REFRESH').collect()
session.sql('DROP TASK IF EXISTS DEMO.ML_DEMO.ML_TRAINING_DAG').collect()

# DAG 정의 및 배포
with DAG(name='ML_TRAINING_DAG', schedule=timedelta(days=1),
         warehouse='COMPUTE_WH') as dag:
    task_prep  = DAGTask('PREPARE_FEATURES', definition='CALL DEMO.ML_DEMO.PREPARE_FEATURES()')
    task_train = DAGTask('TRAIN_MODEL', definition='CALL DEMO.ML_DEMO.TRAIN_MODEL_PROC()')
    task_score = DAGTask('SCORE_CUSTOMERS', definition='CALL DEMO.ML_DEMO.SCORE_CUSTOMERS()')
    task_prep >> task_train >> task_score

schema = root.databases['DEMO'].schemas['ML_DEMO']
DAGOperation(schema).deploy(dag, mode=CreateMode.or_replace)
print('ML_TRAINING_DAG deployed (3 tasks: PREPARE_FEATURES → TRAIN_MODEL → SCORE_CUSTOMERS)')


## 정리

| 방식 | 용도 |
|------|------|
| `@remote` | 개발/수동 실행 (Compute Pool) |
| Task Graph | 자동화/스케줄링 (매일 실행) |
